In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder

In [ ]:
df_full = pd.read_csv("/kaggle/input/competitions/final-project-danit-ds-3-4-2/train.csv")
df_full.shape

In [ ]:
df_full.info()

In [ ]:
df_full.head()

In [ ]:
df_full.describe()

**Аналіз числових ознак**

Age - середній вік 43 роки, аудиторія доросла, переважно середнього віку

Rating - середня оцінка 4.2 з 5, покупці загалом задоволені, 50% відгуків мають оцінку 5 (медіана = 5) — датасет незбалансований.

Recommended - середнє приблизно 0,82, тобто 82% покупців рекомендують товар.Це підтверджує незбалансованість: клас "1" домінує.

Pos_Feedback_Cnt - більшість відгуків мають 0–3 лайки (50% та 75%), але максимум — 122, тобто є викиди (outliers).

In [ ]:
df_full.isna().sum()

In [ ]:
# об'єднуємо текстові колонки
# df_full["Full_Review"] = (df_full["Review_Title"].fillna("") + " " + df_full["Review"].fillna("")).str.strip()

# хочу поставити нову ознаку саме після Review
# col = df_full.pop("Full_Review")
# df_full.insert(4, "Full_Review", col)

# df_full.head(1)

In [ ]:
# df_full.isna().sum()

**Об'єднання стовпчиків схоже майже не має впливу, але хоч один рядок не втрачений.**

**Решту треба видалити, 502 рядки з 14091 — це незначна втрата. Так як це текстові відгуки, а не числа, то їх не можна замінити чимось схожим. Це тільки спотворить загальну картину.**

In [ ]:
# Візуалізація розподілу цільових ознак


fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Rating: 1–5
rating_counts = df_full["Rating"].value_counts().sort_index()
print(rating_counts)

# відступ для написання тексту над графіком
offset = df_full["Rating"].value_counts().max() * 0.02  # 2% від максимуму

axes[0].bar(rating_counts.index, rating_counts.values, color="steelblue")
for idx, val in rating_counts.items():
    axes[0].text(idx, val + offset, f"{val / len(df_full):.1%}", ha="center", fontsize=9)
axes[0].set_title("Розподіл Rating")
axes[0].set_xlabel("Rating")
axes[0].set_ylabel("Кількість")

# Recommended: 0, 1
rec_counts = df_full["Recommended"].value_counts().sort_index()
print(rec_counts)

# відступ для написання тексту над графіком
offset = df_full["Recommended"].value_counts().max() * 0.02  

axes[1].bar(["Не рекомендує (0)", "Рекомендує (1)"], rec_counts.values, color=["tomato", "seagreen"])
for i, val in enumerate(rec_counts.values):
    axes[1].text(i, val + offset, f"{val / len(df_full):.1%}", ha="center", fontsize=9)
axes[1].set_title("Розподіл Recommended")
axes[1].set_ylabel("Кількість")

plt.tight_layout()
plt.show()

50% усіх відгуків є позитивними. Класи дуже не збалансовані, треба оов'язково застосовувати class_weight або інші способи зменшити дисбаланс класів.

In [ ]:
# Числові колонки для матриці кореляцій
numeric_cols = ["Age", "Pos_Feedback_Cnt", "Rating", "Recommended"]

df_corr = df_full[numeric_cols].copy()

corr = df_corr.corr()
sns.heatmap(corr,annot=True, fmt=".2f",cmap="coolwarm", square=True,linewidths=0.5)
plt.title("Кореляційна матриця")
plt.tight_layout()
plt.show()

Rating і Recommended мають високу кореляцію - 0,79, якщо будувати дві окремі моделі, вони будуть передбачати частково одне й те саме.

In [ ]:
# Розподіл Age, гістограма + KDE

print(df_full["Age"].unique())

plt.hist(df_full["Age"], bins=30, color="steelblue", edgecolor="white", density=True, alpha=0.7)
df_full["Age"].plot(kind="kde",color="navy")
plt.title ("Розподіл Age")
plt.xlabel("Вік")

plt.tight_layout()
plt.show()

розподіл близький до нормального зі зсувом вправо (аудиторія 35–55 років)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, col in zip(axes, ["Division", "Department", "Product_Category"]): # поєднуємо попарно axes та назви ознак
    counts = df_full[col].value_counts()
    ax.barh(counts.index, counts.values, color="steelblue")
    ax.set_title(f"Розподіл {col}")
    ax.set_xlabel("Кількість")

plt.tight_layout()
plt.show()

print(df_full["Division"].value_counts())

General           8062 -  основна жіноча лінія — повсякденний та діловий одяг

General Petite    4657 - те саме, але для невисоких жінок (petite = до ~163 см)

Intimates          860 - нижня білизна та домашній одяг


Я так розумію, що Division, Department, Product_Category — це вкладені категорії, тобто вони несуть частково одну й ту саму інформацію. Включати всі три одночасно — означає додавати мультиколінеарність. Мабуть варто вибрати найінформативніший рівень або подумати як їх використовувати надалі.

In [ ]:
# Спочатку я робила в кожній клітинці окремо обробку датасету. Потім зрозуміла, що те ж саме треба робити і для тестового набору.
# Буде одна функція  для попередньої обробки датасетів, якщо тренувальний - то видаляємо пусті рядки в Full_Review 
# та повністю Id. Для тестового датасету цього робити не можна.

def preprocess(df, is_train=True):
    
    df = df.copy()  # робимо копію, щоб не змінювати оригінал

    # об'єднуємо текстові колонки
    df["Full_Review"] = (df["Review_Title"].fillna("") + " " + df["Review"].fillna("")).str.strip()

    # порожні рядки заповнюємо на NaN, щоб можна було їх видалити
    df["Full_Review"] = df["Full_Review"].replace("", pd.NA)

    # обробка рядків без тексту: різна логіка для train та test
    if is_train:
        df = df.dropna(subset=["Full_Review"])
    else:
        # не видаляємо для тестового датасету
        df["Full_Review"] = df["Full_Review"].fillna("no review")

    # заповнюємо пропуски в категоріальних ознаках
    for col in ["Division", "Department", "Product_Category"]:
        df[col] = df[col].fillna("Unknown")

    # прибираємо Id лише з train
    if is_train:
        df = df.drop("Id", axis=1)

    return df

In [ ]:
df_train_clean = preprocess(df_full, is_train=True)

df_train_clean.head()

In [ ]:
df_train_clean.shape

In [ ]:
df_test  = pd.read_csv("/kaggle/input/competitions/final-project-danit-ds-3-4-2/test.csv")
df_test.shape

In [ ]:
df_test_clean = preprocess(df_test, is_train=False)

df_test_clean.head()

In [ ]:
df_train_clean.to_csv("/kaggle/working/df_train_clean.csv", index=False)
df_test_clean.to_csv("/kaggle/working/df_test_clean.csv", index=False)